# tensorcas — GPT-2 fine-tuning benchmark

4-way storage comparison for GPT-2 fine-tuning checkpoint sequences.
GPT-2 is a native HuggingFace Transformers model — `save_pretrained` is
the standard way users save it, making this the most realistic comparison
for transformer fine-tuning workflows.

Unlike ResNet-18, GPT-2 uses **LayerNorm** (not BatchNorm). LayerNorm has
no running statistics buffers — frozen layers are truly frozen. This means
tensorcas's no-op rate for frozen-backbone fine-tuning approaches 100%, vs the
~49% ceiling seen with ResNet-18.

**Methods compared:**

| Method | What it does |
|---|---|
| `save_pretrained` (pytorch) | HF default — saves `pytorch_model.bin` (pickle-based) |
| `save_pretrained` (safetensors) | HF modern format — saves `model.safetensors` |
| DVC (real) | File-level versioning — tracks the weights file per checkpoint |
| tensorcas | Tensor-level dedup — frozen layers stored once, only changed tensors written |

**Scenarios:**

1. **Full fine-tune** — all layers train for 10 epochs. Baseline: every tensor changes every step.
2. **Frozen backbone** — freeze all transformer blocks, train only the LM head. LayerNorm means frozen layers are truly frozen — no silent buffer updates.
3. **Multi-run sweep** — 4 runs with different seeds and learning rates, shared tensorcas store. Tests cross-run savings when the frozen backbone is identical across runs.

In [ ]:
!pip install -q git+https://github.com/Olamyy/tensorcas.git@hash-cache-no-op-path zstandard torch transformers safetensors dvc

In [ ]:
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Config

sys.path.insert(0, str(Path(".").resolve()))
from utils import tensorcas_bytes, _fmt_bytes

from tensorcas.store import TensorCasStore
from tensorcas.adapters.pytorch import PyTorchAdapter

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print("Imports OK")

## Shared helpers

In [ ]:
def dvc_init(repo_dir: Path, remote_dir: Path) -> None:
    repo_dir.mkdir(parents=True, exist_ok=True)
    remote_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q"], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.email", "bench@tensorcas"], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.name", "Pale Bench"], cwd=repo_dir, check=True)
    subprocess.run(["dvc", "init", "-q"], cwd=repo_dir, check=True)
    subprocess.run(
        ["dvc", "remote", "add", "-d", "local", str(remote_dir)],
        cwd=repo_dir, check=True,
    )


def dvc_track_and_push(repo_dir: Path, path: Path) -> None:
    # Track the epoch directory (not the file) — DVC handles directories cleanly
    # and avoids .gitignore conflicts that occur when adding individual files
    # inside the same parent dir across multiple calls.
    rel_path = path.parent.relative_to(repo_dir)
    subprocess.run(["dvc", "add", str(rel_path)], cwd=repo_dir, check=True)
    subprocess.run(["dvc", "push"], cwd=repo_dir, check=True, capture_output=True)


def dvc_cache_bytes(remote_dir: Path) -> int:
    return sum(p.stat().st_size for p in remote_dir.rglob("*") if p.is_file())


def _weights_bytes(path: Path) -> int:
    """Sum all weight files — handles sharded and non-sharded HF output."""
    total = 0
    for pattern in ("*.bin", "*.safetensors"):
        total += sum(p.stat().st_size for p in path.glob(pattern))
    return total


def save_pretrained_pt(model: nn.Module, path: Path) -> int:
    """Save with save_pretrained pytorch format. Returns weight file size in bytes."""
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path, safe_serialization=False)
    return _weights_bytes(path)


def save_pretrained_st(model: nn.Module, path: Path) -> int:
    """Save with save_pretrained safetensors format. Returns weight file size in bytes."""
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path, safe_serialization=True)
    return _weights_bytes(path)


def print_comparison(
    label: str,
    n_checkpoints: int,
    pt_b: int,
    st_b: int,
    dvc_b: int,
    pale_b: int,
) -> None:
    baseline = pt_b or st_b  # fall back to safetensors if pytorch baseline missing

    def _s(b):
        return f"{(baseline - b) / baseline * 100:.1f}%" if baseline else "—"

    print(f"\n{label}")
    print(f"  Checkpoints : {n_checkpoints}")
    print(f"  {'Method':<28} {'Bytes':>10} {'vs save_pretrained':>20}")
    print(f"  {'-'*28} {'-'*10} {'-'*20}")
    print(f"  {'save_pretrained (pytorch)':<28} {_fmt_bytes(pt_b):>10} {'—':>20}")
    print(f"  {'save_pretrained (safetensors)':<28} {_fmt_bytes(st_b):>10} {_s(st_b):>20}")
    print(f"  {'DVC (real)':<28} {_fmt_bytes(dvc_b):>10} {_s(dvc_b):>20}")
    print(f"  {'tensorcas':<28} {_fmt_bytes(pale_b):>10} {_s(pale_b):>20}")


print("Helpers OK")

## Model setup

Uses a small GPT-2 configuration (6 layers, 6 heads, 384 hidden dim) to keep
training fast on Colab. This is ~30M parameters — large enough to be representative
but small enough to train 10 epochs in a few minutes.

To use the full GPT-2 (117M params), change `use_small=False` in the config cell.

In [ ]:
# Set use_small=False to use full GPT-2 (117M params, slower)
USE_SMALL = True

if USE_SMALL:
    # Small GPT-2: 6 layers, 6 heads, 384 hidden
    config = GPT2Config(
        n_layer=6, n_head=6, n_embd=384,
        vocab_size=50257, n_positions=256,
    )
    MODEL_LABEL = "GPT-2 small (6L/384H)"
else:
    # Full GPT-2
    config = GPT2Config()
    MODEL_LABEL = "GPT-2 (117M)"


def _make_model(seed: int = 42) -> GPT2LMHeadModel:
    torch.manual_seed(seed)
    model = GPT2LMHeadModel(config)
    return model.to(device)


sample = _make_model()
n_params = sum(p.numel() for p in sample.parameters())
n_tensors = len(sample.state_dict())
tensor_bytes = sum(v.numel() * v.element_size() for v in sample.state_dict().values())
print(f"{MODEL_LABEL}")
print(f"  Parameters : {n_params:,}")
print(f"  State dict : {n_tensors} tensors, {_fmt_bytes(tensor_bytes)} uncompressed")
del sample

## Training helpers

In [ ]:
SEQ_LEN = 64
BATCH_SIZE = 8
N_BATCHES = 16   # steps per epoch


def _make_loader(seed: int):
    """Synthetic token sequences for language modelling."""
    rng = np.random.default_rng(seed)
    tokens = torch.from_numpy(
        rng.integers(0, config.vocab_size, (N_BATCHES * BATCH_SIZE, SEQ_LEN)).astype(np.int64)
    ).to(device)
    ds = torch.utils.data.TensorDataset(tokens)
    return torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)


def train_epochs(
    model: nn.Module,
    loader,
    n_epochs: int,
    lr: float,
) -> list[dict]:
    """Train for n_epochs. Returns one state dict per epoch."""
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    state_dicts = []
    model.train()
    for _ in range(n_epochs):
        for (xb,) in loader:
            optimizer.zero_grad()
            loss = model(xb, labels=xb).loss
            loss.backward()
            optimizer.step()
        state_dicts.append({k: v.clone().cpu() for k, v in model.state_dict().items()})
    return state_dicts


print("Training helpers OK")

---
## Scenario 1 — Full fine-tune

All layers train for 10 epochs. Every tensor in the state dict changes every epoch.
This is the worst case for tensorcas — no-op rate is 0%, no cross-step reuse.

Storage comparison is still meaningful: `save_pretrained` stores the full model
every epoch. tensorcas writes every tensor every step (no no-ops), but zstd compression
at the chunk level may still save space vs the pickle-based `.bin` format.

**Expected:** All four methods store roughly the same amount — 10 full copies
of the model. tensorcas has no advantage here.

In [ ]:
N_EPOCHS_S1 = 10

print(f"Training {MODEL_LABEL} — full fine-tune ({N_EPOCHS_S1} epochs)...", end=" ", flush=True)
t0 = time.time()
model_s1 = _make_model(seed=42)
loader_s1 = _make_loader(seed=42)
s1_state_dicts = train_epochs(model_s1, loader_s1, N_EPOCHS_S1, lr=1e-4)
print(f"{time.time() - t0:.1f}s")

In [ ]:
# Measure no-op rate: what fraction of tensors are unchanged between consecutive steps?
from tensorcas.hashing import hash_chunk as _hash
from tensorcas.serialization import tensor_to_bytes as _tensor_to_bytes

def noop_rate(state_dicts: list[dict]) -> float:
    identical = total = 0
    for prev, curr in zip(state_dicts, state_dicts[1:]):
        for k in prev:
            raw_p, _, _ = _tensor_to_bytes(prev[k].numpy())
            raw_c, _, _ = _tensor_to_bytes(curr[k].numpy())
            total += 1
            if _hash(raw_p) == _hash(raw_c):
                identical += 1
    return identical / total * 100 if total else 0.0

s1_noop_rate = noop_rate(s1_state_dicts)
print(f"Scenario 1 no-op rate: {s1_noop_rate:.1f}%  (expected: ~0% — all layers train)")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    tensorcas_root  = tmp / "tensorcas"
    pt_base    = dvc_repo / "pt"
    st_base    = tmp / "st"

    dvc_init(dvc_repo, dvc_remote)
    pt_base.mkdir(parents=True); st_base.mkdir()

    pt_total = st_total = 0
    m = _make_model(seed=42)

    for epoch, sd in enumerate(s1_state_dicts, 1):
        m.load_state_dict(sd)

        pt_path = pt_base / f"epoch_{epoch:02d}"
        pt_total += save_pretrained_pt(m, pt_path)

        st_path = st_base / f"epoch_{epoch:02d}"
        st_total += save_pretrained_st(m, st_path)

        dvc_track_and_push(dvc_repo, pt_path / "pytorch_model.bin")

    m2 = _make_model(seed=42)
    with TensorCasStore(root=tensorcas_root, run_id="full_finetune", adapter=PyTorchAdapter()) as store:
        for epoch, sd in enumerate(s1_state_dicts, 1):
            m2.load_state_dict(sd)
            store.save(m2, step=epoch)

    s1_pt_b  = pt_total
    s1_st_b  = st_total
    s1_dvc_b = dvc_cache_bytes(dvc_remote)
    s1_tc_b  = tensorcas_bytes(tensorcas_root)

print_comparison(
    f"Scenario 1 — Full fine-tune ({N_EPOCHS_S1} epochs)",
    N_EPOCHS_S1, s1_pt_b, s1_st_b, s1_dvc_b, s1_tc_b,
)

---
## Scenario 2 — Frozen backbone fine-tune

The transformer blocks (`transformer.h.*`) are frozen. Only the embeddings
(`transformer.wte`, `transformer.wpe`) remain trainable.

**GPT-2 weight tying:** `lm_head.weight` is the same tensor as `transformer.wte.weight`
— there is no independent LM head parameter. Freezing "everything except lm_head"
leaves the optimizer with an empty parameter list. The correct freeze is
`transformer.h.*` only, which keeps the embeddings (and by extension the tied LM head)
trainable.

**Why frozen blocks still demonstrate tensorcas's advantage:**
The transformer blocks make up the overwhelming majority of model parameters.
With 6 layers frozen, ~95%+ of tensors are unchanged between checkpoints.
tensorcas's no-op path fires for every frozen tensor — only the embedding tensors
are written per step.

**Expected:** High no-op rate (~95%+). tensorcas stores the frozen transformer blocks
once; `save_pretrained` and DVC store the full model every epoch regardless.

In [ ]:
N_EPOCHS_S2 = 10

model_s2 = _make_model(seed=42)

# GPT-2 ties lm_head.weight to transformer.wte.weight — there is no independent
# lm_head parameter. Freeze only the transformer blocks (attention + MLP + LayerNorm),
# leaving the embeddings (wte, wpe) and lm_head unfrozen. In practice this means
# only wte / wpe are trained (lm_head is the same tensor as wte).
for name, param in model_s2.named_parameters():
    if name.startswith("transformer.h."):   # transformer blocks only
        param.requires_grad_(False)

trainable = sum(p.numel() for p in model_s2.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model_s2.parameters())
print(f"Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)")
print(f"Frozen tensors in state dict: {sum(1 for n, p in model_s2.named_parameters() if not p.requires_grad)}")
print()

print(f"Training frozen backbone ({N_EPOCHS_S2} epochs)...", end=" ", flush=True)
t0 = time.time()
loader_s2 = _make_loader(seed=42)
s2_state_dicts = train_epochs(model_s2, loader_s2, N_EPOCHS_S2, lr=1e-4)
print(f"{time.time() - t0:.1f}s")

In [ ]:
s2_noop_rate = noop_rate(s2_state_dicts)
print(f"Scenario 2 no-op rate: {s2_noop_rate:.1f}%  (expected: ~100% for frozen layers)")

# Breakdown by tensor
print("\nPer-tensor breakdown (first 5 changed, first 5 frozen):")
prev, curr = s2_state_dicts[0], s2_state_dicts[1]
changed = []
frozen  = []
for k in prev:
    raw_p, _, _ = _tensor_to_bytes(prev[k].numpy())
    raw_c, _, _ = _tensor_to_bytes(curr[k].numpy())
    if _hash(raw_p) == _hash(raw_c):
        frozen.append(k)
    else:
        changed.append(k)

print(f"\n  Changed ({len(changed)} tensors):")
for k in changed[:5]:
    print(f"    {k}")
print(f"\n  Frozen ({len(frozen)} tensors):")
for k in frozen[:5]:
    print(f"    {k}")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    tensorcas_root  = tmp / "tensorcas"
    pt_base    = dvc_repo / "pt"
    st_base    = tmp / "st"

    dvc_init(dvc_repo, dvc_remote)
    pt_base.mkdir(parents=True); st_base.mkdir()

    pt_total = st_total = 0
    m = _make_model(seed=42)
    for name, param in m.named_parameters():
        if name.startswith("transformer.h."):
            param.requires_grad_(False)

    for epoch, sd in enumerate(s2_state_dicts, 1):
        m.load_state_dict(sd)

        pt_path = pt_base / f"epoch_{epoch:02d}"
        pt_total += save_pretrained_pt(m, pt_path)

        st_path = st_base / f"epoch_{epoch:02d}"
        st_total += save_pretrained_st(m, st_path)

        dvc_track_and_push(dvc_repo, pt_path / "pytorch_model.bin")

    m2 = _make_model(seed=42)
    for name, param in m2.named_parameters():
        if name.startswith("transformer.h."):
            param.requires_grad_(False)

    with TensorCasStore(root=tensorcas_root, run_id="frozen_backbone", adapter=PyTorchAdapter()) as store:
        for epoch, sd in enumerate(s2_state_dicts, 1):
            m2.load_state_dict(sd)
            store.save(m2, step=epoch)
        s2_stats = store.stats()

    s2_pt_b  = pt_total
    s2_st_b  = st_total
    s2_dvc_b = dvc_cache_bytes(dvc_remote)
    s2_tc_b  = tensorcas_bytes(tensorcas_root)

print_comparison(
    f"Scenario 2 — Frozen backbone ({N_EPOCHS_S2} epochs, embeddings only)",
    N_EPOCHS_S2, s2_pt_b, s2_st_b, s2_dvc_b, s2_tc_b,
)
print()
print(f"  tensorcas total chunks  : {s2_stats['total_chunks']}")
print(f"  tensorcas unique chunks : {s2_stats['unique_chunks']}")
print(f"  Dedup ratio        : {s2_stats['dedup_ratio']:.3f}  ({(1 - s2_stats['dedup_ratio'])*100:.0f}% reuse)")

---
## Scenario 3 — Multi-run sweep

4 fine-tuning runs from the same base model (same random seed), varying
learning rate. Backbone fully frozen — only the LM head trains.

All 4 runs use a shared tensorcas store. Because the backbone is frozen and
starts from the same weights, all backbone tensors are identical across runs.
Unlike ResNet-18, there are no BatchNorm running stats to diverge — the
frozen layers are exactly identical in all runs.

**Expected:**
- `save_pretrained` / DVC: stores 4 × 10 = 40 complete model files
- tensorcas: stores the frozen backbone once (run 1); each subsequent run writes
  only the LM head weights unique to that run

In [ ]:
N_EPOCHS_S3 = 10

RUNS = [
    {"run_id": "run_a", "seed": 42,  "lr": 1e-4},
    {"run_id": "run_b", "seed": 42,  "lr": 5e-5},
    {"run_id": "run_c", "seed": 42,  "lr": 2e-4},
    {"run_id": "run_d", "seed": 42,  "lr": 1e-5},
]

print(f"Runs: {len(RUNS)}")
print(f"Total checkpoints: {len(RUNS) * N_EPOCHS_S3}")

In [ ]:
s3_state_dicts = {}

for run in RUNS:
    t0 = time.time()
    print(f"  {run['run_id']} (lr={run['lr']})...", end=" ", flush=True)

    model = _make_model(seed=run["seed"])
    for name, param in model.named_parameters():
        if name.startswith("transformer.h."):
            param.requires_grad_(False)

    loader = _make_loader(seed=run["seed"])
    s3_state_dicts[run["run_id"]] = train_epochs(model, loader, N_EPOCHS_S3, run["lr"])
    print(f"{time.time() - t0:.1f}s")

print(f"\nTotal checkpoints: {sum(len(v) for v in s3_state_dicts.values())}")

In [ ]:
s3_noop_rates = {}
for run in RUNS:
    s3_noop_rates[run["run_id"]] = noop_rate(s3_state_dicts[run["run_id"]])
    print(f"  {run['run_id']} no-op rate: {s3_noop_rates[run['run_id']]:.1f}%")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    tensorcas_root  = tmp / "tensorcas"
    pt_base    = dvc_repo / "pt"
    st_base    = tmp / "st"

    dvc_init(dvc_repo, dvc_remote)
    pt_base.mkdir(parents=True); st_base.mkdir()

    pt_total = st_total = 0
    s3_tensorcas_size_after = {}
    n_checkpoints = 0

    for run in RUNS:
        run_id = run["run_id"]
        (pt_base / run_id).mkdir(); (st_base / run_id).mkdir()

        m = _make_model(seed=run["seed"])
        for name, param in m.named_parameters():
            if name.startswith("transformer.h."):
                param.requires_grad_(False)

        for epoch, sd in enumerate(s3_state_dicts[run_id], 1):
            m.load_state_dict(sd)

            pt_path = pt_base / run_id / f"epoch_{epoch:02d}"
            pt_total += save_pretrained_pt(m, pt_path)

            st_path = st_base / run_id / f"epoch_{epoch:02d}"
            st_total += save_pretrained_st(m, st_path)

            dvc_track_and_push(dvc_repo, pt_path / "pytorch_model.bin")
            n_checkpoints += 1

        with TensorCasStore(root=tensorcas_root, run_id=run_id, adapter=PyTorchAdapter()) as store:
            m2 = _make_model(seed=run["seed"])
            for name, param in m2.named_parameters():
                if name.startswith("transformer.h."):
                    param.requires_grad_(False)
            for epoch, sd in enumerate(s3_state_dicts[run_id], 1):
                m2.load_state_dict(sd)
                store.save(m2, step=epoch)

        s3_tensorcas_size_after[run_id] = tensorcas_bytes(tensorcas_root)
        print(f"  {run_id}: done")

    s3_pt_b  = pt_total
    s3_st_b  = st_total
    s3_dvc_b = dvc_cache_bytes(dvc_remote)
    s3_tc_b  = tensorcas_bytes(tensorcas_root)

print_comparison(
    f"Scenario 3 — Multi-run sweep ({len(RUNS)} runs × {N_EPOCHS_S3} epochs)",
    n_checkpoints, s3_pt_b, s3_st_b, s3_dvc_b, s3_tc_b,
)

print("\ntensorcas store size after each run:")
prev = 0
for run_id, size in s3_tensorcas_size_after.items():
    print(f"  {run_id:<8} {_fmt_bytes(size):>10}  (+{_fmt_bytes(size - prev)})")
    prev = size

---
## Summary

| Scenario | Ckpts | save_pretrained (pt) | save_pretrained (st) | DVC | tensorcas | tensorcas vs save_pretrained |
|---|---|---|---|---|---|---|
| 1 — Full fine-tune (10 epochs) | 10 | TBD | TBD | TBD | TBD | TBD |
| 2 — Frozen backbone (10 epochs) | 10 | TBD | TBD | TBD | TBD | TBD |
| 3 — Multi-run sweep (4 runs × 10 epochs) | 40 | TBD | TBD | TBD | TBD | TBD |

**Why GPT-2 is a better story than ResNet-18:**

ResNet-18's BatchNorm buffers (`running_mean`, `running_var`) update every forward
pass in `model.train()` mode, even for frozen layers — capping tensorcas's no-op rate at
~49% instead of the theoretical ~100%.

GPT-2 uses LayerNorm, which has **no running statistics buffers**. When a layer is
frozen with `requires_grad_(False)`, its parameters are genuinely unchanged between
checkpoints. tensorcas's no-op path fires for every frozen tensor — the no-op rate
approaches 100% for the frozen backbone.

**Scenario 1 (full fine-tune):** tensorcas has no advantage. All methods store ~10 full
copies of the model. This is the expected result — tensorcas is not designed for full
fine-tune workflows where everything changes.

**Scenario 2 (frozen backbone):** tensorcas stores the frozen transformer blocks once.
Only the LM head is written per step. `save_pretrained` and DVC store the full model
every epoch regardless.

**Scenario 3 (multi-run):** tensorcas stores the frozen backbone once across all 4 runs.
Because there are no BatchNorm running stats, the frozen layers are byte-identical
across all runs — cross-run sharing is close to 100% for the frozen portion.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 150})

from pathlib import Path
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

MB = 1024**2

# --- Figure 1: Storage comparison — 4 methods × 3 scenarios ---
# Read from s1_*, s2_*, s3_* variables computed in the scenario cells above
scenarios = ["Full fine-tune\n(10 epochs)", "Frozen backbone\n(10 epochs)", "Multi-run sweep\n(4×10 epochs)"]
pt_mb        = [s1_pt_b / MB,  s2_pt_b / MB,  s3_pt_b / MB]
st_mb        = [s1_st_b / MB,  s2_st_b / MB,  s3_st_b / MB]
dvc_mb       = [s1_dvc_b / MB, s2_dvc_b / MB, s3_dvc_b / MB]
tensorcas_mb = [s1_tc_b / MB,  s2_tc_b / MB,  s3_tc_b / MB]

x = range(len(scenarios))
width = 0.2
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - 1.5*width for i in x], pt_mb,        width, label="save_pretrained (pt)", color="#4C72B0")
ax.bar([i - 0.5*width for i in x], st_mb,        width, label="save_pretrained (st)", color="#8172B2")
ax.bar([i + 0.5*width for i in x], dvc_mb,       width, label="DVC",                  color="#DD8452")
ax.bar([i + 1.5*width for i in x], tensorcas_mb, width, label="TensorCas",             color="#55A868")
ax.set_xticks(list(x))
ax.set_xticklabels(scenarios)
ax.set_ylabel("Total storage (MB)")
ax.set_title("Storage comparison: 4 methods × 3 GPT-2 fine-tuning scenarios")
ax.legend()
ax.set_yscale("log")
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g} MB"))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "gpt2_storage_comparison.png")
plt.show()
print("Saved figures/gpt2_storage_comparison.png")

# --- Figure 2: No-op rate by scenario ---
# s3 noop rate: average across all runs
s3_avg_noop = sum(s3_noop_rates.values()) / len(s3_noop_rates)
noop_scenarios = ["Full fine-tune", "Frozen backbone", f"Multi-run sweep\n(avg, {len(RUNS)} runs)"]
noop_rate_vals = [s1_noop_rate, s2_noop_rate, s3_avg_noop]

colors = ["#C44E52" if r < 50 else "#55A868" for r in noop_rate_vals]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(noop_scenarios, noop_rate_vals, color=colors, width=0.5)
for bar, val in zip(bars, noop_rate_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("No-op rate (%)")
ax.set_title("No-op rate by scenario (GPT-2)\n(LayerNorm — no silent buffer updates in frozen layers)")
ax.set_ylim(0, 120)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "gpt2_noop_rate.png")
plt.show()
print("Saved figures/gpt2_noop_rate.png")

# --- Figure 3: Multi-run marginal cost (scenario 3) ---
# Read from s3_tensorcas_size_after collected in scenario 3 cell
run_labels = list(s3_tensorcas_size_after.keys())
cumulative_mb_vals = [v / MB for v in s3_tensorcas_size_after.values()]
prev_vals = [0] + list(s3_tensorcas_size_after.values())[:-1]
marginal_vals = [cur - prev for cur, prev in zip(s3_tensorcas_size_after.values(), prev_vals)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(run_labels, cumulative_mb_vals, color="#55A868", width=0.5)
ax.set_ylabel("Cumulative store size (MB)")
ax.set_title("GPT-2 multi-run sweep — TensorCas marginal cost per run\n(LayerNorm: frozen layers are truly frozen)")
ax.set_ylim(0, max(cumulative_mb_vals) * 1.3)
for i, (mb, mg) in enumerate(zip(cumulative_mb_vals, marginal_vals)):
    label = f"+{mg/1024:.0f} KB" if mg < MB else f"+{mg/MB:.1f} MB"
    ax.text(i, mb + max(cumulative_mb_vals) * 0.02, label, ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "gpt2_marginal_cost.png")
plt.show()
print("Saved figures/gpt2_marginal_cost.png")